This notebook performs a structured analysis of gendered word pairs in Open English WordNet. It loads the RDF version of WordNet and extracts semantic relations (e.g. hypernyms, hyponyms, and other WordNet-specific relations) for a manually curated set of gendered terms. The goal is to compare the semantic structure of male and female terms and identify potential asymmetries in their representation.

In [3]:
from rdflib import Graph, Namespace
from collections import defaultdict
import pandas as pd

g = Graph()
g.parse("english-wordnet-2025.ttl", format="turtle")

ONTOLEX = Namespace("http://www.w3.org/ns/lemon/ontolex#")
WN = Namespace("https://globalwordnet.github.io/schemas/wn#")


The list of gendered word pairs is adapted from Lu et al. (2019), but modified for use with WordNet. Plural forms and animal terms were removed to keep the focus on human-related concepts, and uncommon or outdated forms (e.g. murderess, beau) were excluded to avoid noise. Some terms were also replaced with more precise WordNet-compatible versions, such as male monarch instead of king, to ensure that each term maps to a single, unambiguous synset.

The final set includes 21 pairs across different domains, including social roles, occupations, family relations, religion, and status. This diversity allows for comparing gendered terms in different contexts, while the manual selection means the analysis should be understood as a focused case study rather than a complete representation of gender bias in WordNet.

In [ ]:
pairs = [
    ("male monarch", "female monarch"),
    ("prince", "princess"),
    ("emperor", "empress"),
    ("actor", "actress"),
    ("waiter", "waitress"),
    ("businessman", "businesswoman"),
    ("policeman", "policewoman"),
    ("congressman", "congresswoman"),
    ("male parent", "female parent"),
    ("son", "daughter"),
    ("male sibling", "female sibling"),
    ("uncle", "aunt"),
    ("nephew", "niece"),
    ("grandfather", "grandmother"),
    ("grandson", "granddaughter"),
    ("husband", "wife"),
    ("boyfriend", "lady friend"),
    ("monk", "nun"),
    ("priest", "priestess"),
    ("abbot", "abbess"),
    ("god", "goddess"),
]

Then, for each term, a specific synset URI was manually selected from Open English WordNet. This step was necessary because many words have multiple meanings (polysemy), and automatic selection would introduce noise or mismatches (e.g. emperor as a butterfly vs. a ruler). By choosing one synset per term, we ensure that each pair is semantically comparable.

In [ ]:

chosen_synsets = {
    "male monarch": "https://en-word.net/id/oewn-10251212-n",
    "female monarch": "https://en-word.net/id/oewn-10518940-n",
    "prince": "https://en-word.net/id/oewn-10492384-n",
    "princess": "https://en-word.net/id/oewn-10493649-n",
    "emperor": "https://en-word.net/id/oewn-10072812-n",
    "empress": "https://en-word.net/id/oewn-10073247-n",
    "actor": "https://en-word.net/id/oewn-09784701-n",
    "actress": "https://en-word.net/id/oewn-09787123-n",
    "waiter": "https://en-word.net/id/oewn-10783051-n",
    "waitress": "https://en-word.net/id/oewn-10783288-n",
    "businessman": "https://en-word.net/id/oewn-09901459-n",
    "businesswoman": "https://en-word.net/id/oewn-09902067-n",
    "policeman": "https://en-word.net/id/oewn-80600405-n",
    "policewoman": "https://en-word.net/id/oewn-10468986-n",
    "congressman": "https://en-word.net/id/oewn-83115757-n",
    "congresswoman": "https://en-word.net/id/oewn-80278004-n",
    "male parent": "https://en-word.net/id/oewn-10100638-n",
    "female parent": "https://en-word.net/id/oewn-10352098-n",
    "son": "https://en-word.net/id/oewn-10643436-n",
    "daughter": "https://en-word.net/id/oewn-10012375-n",
    "male sibling": "https://en-word.net/id/oewn-10305781-n",
    "female sibling": "https://en-word.net/id/oewn-10103950-n",
    "uncle": "https://en-word.net/id/oewn-10755748-n",
    "aunt": "https://en-word.net/id/oewn-09842904-n",
    "nephew": "https://en-word.net/id/oewn-10373054-n",
    "niece": "https://en-word.net/id/oewn-10377312-n",
    "grandfather": "https://en-word.net/id/oewn-10161911-n",
    "grandmother": "https://en-word.net/id/oewn-10162267-n",
    "grandson": "https://en-word.net/id/oewn-10162819-n",
    "granddaughter": "https://en-word.net/id/oewn-10161252-n",
    "husband": "https://en-word.net/id/oewn-10213586-n",
    "wife": "https://en-word.net/id/oewn-10800308-n",
    "boyfriend": "https://en-word.net/id/oewn-09890770-n",
    "lady friend": "https://en-word.net/id/oewn-10150206-n",
    "monk": "https://en-word.net/id/oewn-10131898-n",
    "nun": "https://en-word.net/id/oewn-10387708-n",
    "priest": "https://en-word.net/id/oewn-10490364-n",
    "priestess": "https://en-word.net/id/oewn-10491155-n",
    "abbot": "https://en-word.net/id/oewn-09773735-n",
    "abbess": "https://en-word.net/id/oewn-09773548-n",
    "god": "https://en-word.net/id/oewn-09528550-n",
    "goddess": "https://en-word.net/id/oewn-09558733-n",
}

We restrict the analysis to a predefined set of WordNet-specific semantic relations (e.g. hypernym, hyponym, meronymy, entailment, similarity). This ensures that only linguistically meaningful connections within the WordNet schema are counted, rather than all possible RDF predicates. By doing so, we obtain a cleaner and more comparable measure of semantic structure across terms. At the same time, this choice limits the analysis, as other potentially relevant relations (e.g. lexical or external links) are excluded.

In [ ]:
WN_RELATIONS = [
    "also",
    "antonym",
    "attribute",
    "causes",
    "domain_region",
    "domain_topic",
    "entails",
    "exemplifies",
    "has_domain_region",
    "has_domain_topic",
    "holo_member",
    "holo_part",
    "holo_substance",
    "hypernym",
    "hyponym",
    "is_caused_by",
    "is_entailed_by",
    "is_exemplified_by",
    "mero_member",
    "mero_part",
    "mero_substance",
    "participle",
    "pertainym",
    "similar",
]

This part defines the SPARQL helper functions used to extract the structural information for each selected synset. timed_query() runs a SPARQL query and prints how long it took, which is useful because some queries, especially hyponym queries, can be slow. The hypernym and hyponym functions retrieve the related synsets and then map them back to readable word labels through OntoLex lexical entries. The relation-count functions measure how many selected WordNet relations are attached to each synset, either only outgoing or in both directions.

In [ ]:
import time

def timed_query(graph, query, label):
    start = time.time()
    print(f"    -> {label} ...", flush=True)
    rows = list(graph.query(query))
    elapsed = time.time() - start
    print(f"       done in {elapsed:.2f}s ({len(rows)} rows)", flush=True)
    return rows

In [ ]:
def get_hypernyms_sparql(graph, synset_uri):
    """
    Given an RDF graph (WordNet) and a synset URI, this function retrieves:
    1. All hypernyms (more general concepts) of the synset
    2. Their human-readable labels (if available)

    Returns:
        - A sorted list of hypernym labels (or URIs if no label exists)
        - The number of unique hypernyms found
    """

    # Build a SPARQL query string dynamically using the given synset URI
    # The query:
    # - Finds all hypernyms of the synset
    # - Tries (OPTIONAL) to get a readable word (writtenRep) for each hypernym
    q = f"""
    PREFIX wn: <https://globalwordnet.github.io/schemas/wn#>
    PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>

    SELECT DISTINCT ?hypernym ?writtenRep
    WHERE {{
      # Get all hypernyms of the given synset
      <{synset_uri}> wn:hypernym ?hypernym .

      # Try to find a human-readable label for each hypernym
      OPTIONAL {{
        # Link hypernym synset to its lexical sense
        ?sense ontolex:isLexicalizedSenseOf ?hypernym .

        # Link sense to its lexical entry and canonical form
        ?entry ontolex:sense ?sense ;
               ontolex:canonicalForm ?form .

        # Extract the actual written word (label)
        ?form ontolex:writtenRep ?writtenRep .
      }}
    }}
    """

    # Execute the query using a helper function ( logs execution time)
    rows = timed_query(graph, q, f"hypernyms for {synset_uri}")

    # Process query results:
    # - If a label (writtenRep) exists, use it
    # - Otherwise fall back to the raw hypernym URI
    # - Remove duplicates using a set
    # - Convert everything to strings
    # - Sort alphabetically (case-insensitive)
    labels = sorted(
        set(
            str(r.writtenRep) if r.writtenRep else str(r.hypernym)
            for r in rows
        ),
        key=str.lower
    )

    # Return both the list of labels and the count of unique hypernyms
    return labels, len(labels)

In [ ]:
def get_hyponyms_sparql(graph, synset_uri):
    """
    Given an RDF graph (WordNet) and a synset URI, this function retrieves:
    1. All hyponyms (more specific concepts) of the synset
    2. Their human-readable labels (if available)

    Returns:
        - A sorted list of hyponym labels (or URIs if no label exists)
        - The number of unique hyponyms found
    """

    # Build a SPARQL query dynamically using the given synset URI
    # The query:
    # - Finds all hyponyms of the synset
    # - Tries (OPTIONAL) to retrieve readable labels (writtenRep)
    q = f"""
    PREFIX wn: <https://globalwordnet.github.io/schemas/wn#>
    PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>

    SELECT DISTINCT ?hyponym ?writtenRep
    WHERE {{
      # Get all hyponyms of the given synset (more specific concepts)
      <{synset_uri}> wn:hyponym ?hyponym .

      # Try to find a human-readable label for each hyponym
      OPTIONAL {{
        # Link hyponym synset to its lexical sense
        ?sense ontolex:isLexicalizedSenseOf ?hyponym .

        # Link sense to its lexical entry and canonical form
        ?entry ontolex:sense ?sense ;
               ontolex:canonicalForm ?form .

        # Extract the actual written word (label)
        ?form ontolex:writtenRep ?writtenRep .
      }}
    }}
    """
    rows = timed_query(graph, q, f"hyponyms for {synset_uri}")

    # Process results:
    # - Prefer readable labels (writtenRep)
    # - Fall back to URI if no label exists
    # - Remove duplicates
    # - Sort alphabetically (case-insensitive)
    labels = sorted(
        set(
            str(r.writtenRep) if r.writtenRep else str(r.hyponym)
            for r in rows
        ),
        key=str.lower
    )

    # Return both the list of labels and the number of unique hyponyms
    return labels, len(labels)

In [ ]:
def get_relation_counts_sparql(graph, synset_uri):
    """
    Given an RDF graph (WordNet) and a synset URI, this function measures
    how "structurally rich" the synset is in terms of its outgoing relations.

    Specifically, it counts:
    1. predicateCount: number of DISTINCT relation types (e.g. hypernym, hyponym, etc.)
    2. edgeCount: total number of outgoing relation edges (including duplicates)

    This is useful for estimating how many different semantic connections
    a synset participates in, and how many total links it has.

    Returns:
        - predicateCount (int): number of unique relation types
        - edgeCount (int): total number of outgoing edges
    """

    # Build a comma-separated list of allowed WordNet relations
    # Example: wn:hypernym, wn:hyponym, wn:meronym, ...
    # This ensures we only count meaningful semantic relations,
    # not all RDF triples attached to the synset
    relation_filter = ",\n        ".join(f"wn:{r}" for r in WN_RELATIONS)

    # SPARQL query:
    # - Select all outgoing triples from the synset (<synset_uri> ?p ?o)
    # - Restrict predicates (?p) to WordNet relations defined above
    # - Count:
    #     * DISTINCT ?p → how many different relation types exist
    #     * total rows     → how many edges exist overall
    q = f"""
    PREFIX wn: <https://globalwordnet.github.io/schemas/wn#>

    SELECT (COUNT(DISTINCT ?p) AS ?predicateCount)
           (COUNT(*) AS ?edgeCount)
    WHERE {{
      # All outgoing edges from this synset
      <{synset_uri}> ?p ?o .

      # Only keep edges where the predicate is a WordNet relation
      FILTER(?p IN (
        {relation_filter}
      ))
    }}
    """
    rows = timed_query(graph, q, f"outgoing relation counts for {synset_uri}")

    # SPARQL returns a single row with aggregated counts
    # If we got a result, extract and convert to integers
    if rows:
        return int(rows[0].predicateCount), int(rows[0].edgeCount)

    # Fallback in case no relations are found or query fails
    return 0, 0

In [ ]:
def get_relation_counts_both_directions_sparql(graph, synset_uri):
    """
    Given an RDF graph (WordNet) and a synset URI, this function counts
    how many semantic relation edges connect to the synset in either direction.

    It counts:
    1. outgoing relations: synset -> other synset
    2. incoming relations: other synset -> synset

    This gives a broader measure of how connected the synset is in the
    WordNet graph.

    Returns:
        - edgeCount (int): total number of incoming and outgoing relation edges
    """

    # Build a comma-separated list of WordNet relations to include.
    # This prevents the query from counting unrelated RDF metadata.
    relation_filter = ",\n        ".join(f"wn:{r}" for r in WN_RELATIONS)

    # SPARQL query:
    # - First block counts outgoing semantic relations from the synset
    # - Second block counts incoming semantic relations to the synset
    # - UNION combines both directions into one result set
    # - COUNT(*) returns the total number of matching edges
    q = f"""
    PREFIX wn: <https://globalwordnet.github.io/schemas/wn#>

    SELECT (COUNT(*) AS ?edgeCount)
    WHERE {{
      {{
        # Outgoing relation:
        # the given synset points to another node
        <{synset_uri}> ?p ?x .

        # Only count selected WordNet relation predicates
        FILTER(?p IN (
          {relation_filter}
        ))
      }}
      UNION
      {{
        # Incoming relation:
        # another node points to the given synset
        ?x ?p <{synset_uri}> .

        # Again, only count selected WordNet relation predicates
        FILTER(?p IN (
          {relation_filter}
        ))
      }}
    }}
    """
    rows = timed_query(graph, q, f"both-direction relation counts for {synset_uri}")

    # Aggregation queries usually return one row.
    # If a row exists, extract the edge count and convert it to an integer.
    if rows:
        return int(rows[0].edgeCount)

    # Fallback value if the query returns no usable result.
    return 0

This is the main extraction loop. It goes through each gendered pair, looks up the manually selected synset URI, and runs the SPARQL helper functions to collect hypernym labels, hyponym labels, and WordNet relation counts. Each result is stored as one row in a dataframe, with one row per term. The progress messages make it easier to see which term is currently being processed and where the code might slow down or fail. At the end, the full result table is saved as a CSV file for later analysis.

In [ ]:
results = []

total_terms = len(pairs) * 2
term_counter = 0

for male_term, female_term in pairs:
    print(f"\nPAIR: {male_term} / {female_term}", flush=True)

    for gender, term in [("male", male_term), ("female", female_term)]:
        term_counter += 1
        synset_uri = chosen_synsets[term]

        print(f"  [{term_counter}/{total_terms}] term: {term}", flush=True)
        print(f"      synset: {synset_uri}", flush=True)

        try:
            hypernym_labels, hypernym_count = get_hypernyms_sparql(g, synset_uri)
            hyponym_labels, hyponym_count = get_hyponyms_sparql(g, synset_uri)
            pred_count, edge_count = get_relation_counts_sparql(g, synset_uri)
            both_dir_count = get_relation_counts_both_directions_sparql(g, synset_uri)

            results.append({
                "pair_male": male_term,
                "pair_female": female_term,
                "gender_side": gender,
                "term": term,
                "synset_uri": synset_uri,
                "hypernym_count": hypernym_count,
                "hypernym_labels": " | ".join(hypernym_labels),
                "hyponym_count": hyponym_count,
                "hyponym_labels": " | ".join(hyponym_labels),
                "wn_relation_predicate_count": pred_count,
                "wn_relation_edge_count": edge_count,
                "wn_relation_both_directions": both_dir_count,
            })

            print(f"      saved row for {term}", flush=True)

        except Exception as e:
            print(f"      ERROR on {term}: {e}", flush=True)
            results.append({
                "pair_male": male_term,
                "pair_female": female_term,
                "gender_side": gender,
                "term": term,
                "synset_uri": synset_uri,
                "hypernym_count": None,
                "hypernym_labels": None,
                "hyponym_count": None,
                "hyponym_labels": None,
                "wn_relation_predicate_count": None,
                "wn_relation_edge_count": None,
                "wn_relation_both_directions": None,
            })

df = pd.DataFrame(results)

print("\nFinished.", flush=True)
print(df.head(), flush=True)
print(df.shape, flush=True)

df.to_csv("wordnet_gender_pairs_clean_sparql.csv", index=False, encoding="utf-8")
print("Saved to wordnet_gender_pairs_clean_sparql.csv", flush=True)


PAIR: male monarch / female monarch
  [1/42] term: male monarch
      synset: https://en-word.net/id/oewn-10251212-n
    -> hypernyms for https://en-word.net/id/oewn-10251212-n ...
       done in 6.13s (5 rows)
    -> hyponyms for https://en-word.net/id/oewn-10251212-n ...
       done in 4.98s (4 rows)
    -> outgoing relation counts for https://en-word.net/id/oewn-10251212-n ...
       done in 0.02s (1 rows)
    -> both-direction relation counts for https://en-word.net/id/oewn-10251212-n ...
       done in 0.04s (1 rows)
      saved row for male monarch
  [2/42] term: female monarch
      synset: https://en-word.net/id/oewn-10518940-n
    -> hypernyms for https://en-word.net/id/oewn-10518940-n ...
       done in 1.21s (1 rows)
    -> hyponyms for https://en-word.net/id/oewn-10518940-n ...
       done in 1.25s (1 rows)
    -> outgoing relation counts for https://en-word.net/id/oewn-10518940-n ...
       done in 0.02s (1 rows)
    -> both-direction relation counts for https://en-word.n

KeyboardInterrupt: 

This part converts the term-level results into a pairwise comparison table. It separates male and female rows, renames the columns, and then merges them back together so that each gendered pair appears in one row. The code then calculates the differences between male and female terms for hypernym counts, hyponym counts, and relation counts. This makes it easier to see whether one side of a pair has a richer WordNet structure than the other. Finally, the pairwise comparison table is saved as a CSV file for analysis and visualization.

In [20]:
import pandas as pd

df = pd.read_csv("wordnet_gender_pairs_clean_sparql.csv")

male_df = df[df["gender_side"] == "male"].copy()
female_df = df[df["gender_side"] == "female"].copy()

male_df = male_df.rename(columns={
    "term": "male_term",
    "synset_uri": "male_synset_uri",
    "hypernym_count": "male_hypernym_count",
    "hypernym_labels": "male_hypernym_labels",
    "hyponym_count": "male_hyponym_count",
    "hyponym_labels": "male_hyponym_labels",
    "wn_relation_predicate_count": "male_relation_predicate_count",
    "wn_relation_edge_count": "male_relation_edge_count",
    "wn_relation_both_directions": "male_relation_both_directions",
})

female_df = female_df.rename(columns={
    "term": "female_term",
    "synset_uri": "female_synset_uri",
    "hypernym_count": "female_hypernym_count",
    "hypernym_labels": "female_hypernym_labels",
    "hyponym_count": "female_hyponym_count",
    "hyponym_labels": "female_hyponym_labels",
    "wn_relation_predicate_count": "female_relation_predicate_count",
    "wn_relation_edge_count": "female_relation_edge_count",
    "wn_relation_both_directions": "female_relation_both_directions",
})

pairwise_df = male_df.merge(
    female_df,
    on=["pair_male", "pair_female"],
    how="inner"
)

pairwise_df["diff_hypernym_count"] = (
    pairwise_df["male_hypernym_count"] - pairwise_df["female_hypernym_count"]
)
pairwise_df["diff_hyponym_count"] = (
    pairwise_df["male_hyponym_count"] - pairwise_df["female_hyponym_count"]
)
pairwise_df["diff_relation_edge_count"] = (
    pairwise_df["male_relation_edge_count"] - pairwise_df["female_relation_edge_count"]
)
pairwise_df["diff_relation_both_directions"] = (
    pairwise_df["male_relation_both_directions"] - pairwise_df["female_relation_both_directions"]
)

cols = [
    "male_term", "female_term",
    "male_hypernym_count", "female_hypernym_count", "diff_hypernym_count",
    "male_hyponym_count", "female_hyponym_count", "diff_hyponym_count",
    "male_relation_edge_count", "female_relation_edge_count", "diff_relation_edge_count",
]
print(pairwise_df[cols].to_string(index=False))

pairwise_df.to_csv("wordnet_gender_pairs_pairwise.csv", index=False, encoding="utf-8")

   male_term    female_term  male_hypernym_count  female_hypernym_count  diff_hypernym_count  male_hyponym_count  female_hyponym_count  diff_hyponym_count  male_relation_edge_count  female_relation_edge_count  diff_relation_edge_count
male monarch female monarch                    5                      1                    4                   4                     1                   3                         6                           3                         3
      prince       princess                    5                      5                    0                  10                     5                   5                        13                           7                         6
     emperor        empress                    3                      3                    0                   9                     0                   9                         7                           2                         5
       actor        actress                    2            

This line sorts the pairwise results by the difference in hyponym counts, from highest to lowest, and prints a compact table with the most relevant columns. This highlights which male–female pairs show the largest asymmetries in the number of more specific concepts (hyponyms), making potential structural biases in WordNet immediately visible.

In [21]:
print(
    pairwise_df.sort_values("diff_hyponym_count", ascending=False)[
        ["male_term", "female_term", "male_hyponym_count", "female_hyponym_count", "diff_hyponym_count"]
    ].to_string(index=False)
)

   male_term    female_term  male_hyponym_count  female_hyponym_count  diff_hyponym_count
       actor        actress                  33                     3                  30
         god        goddess                  29                     2                  27
      priest      priestess                  16                     0                  16
     emperor        empress                   9                     0                   9
      prince       princess                  10                     5                   5
         son       daughter                   6                     1                   5
      waiter       waitress                   6                     2                   4
male monarch female monarch                   4                     1                   3
        monk            nun                   4                     1                   3
      nephew          niece                   2                     2                   0
 businessm

This function is used for manual inspection of the selected synsets. It prints the hypernyms and hyponyms of each term in a readable format, making it easier to verify that the chosen synset corresponds to the intended meaning and to qualitatively explore the semantic structure. Unlike the CSV and JSON outputs, which are used for systematic analysis, this function serves as a debugging and interpretation tool.

In [ ]:
def print_synset_details(graph, term, synset_uri):
    print(f"\n=== {term.upper()} ===")
    print(f"Synset: {synset_uri}")

    hypernyms, _ = get_hypernyms_sparql(graph, synset_uri)
    hyponyms, _ = get_hyponyms_sparql(graph, synset_uri)

    print("\nHypernyms:")
    if hypernyms:
        for h in hypernyms:
            print(f"  • {h}")
    else:
        print("  (none)")

    print("\nHyponyms:")
    if hyponyms:
        for h in hyponyms:
            print(f"  • {h}")
    else:
        print("  (none)")

for term, synset_uri in chosen_synsets.items():
    print_synset_details(g, term, synset_uri)


=== MALE MONARCH ===
Synset: https://en-word.net/id/oewn-10251212-n
    -> hypernyms for https://en-word.net/id/oewn-10251212-n ...
       done in 7.57s (5 rows)
    -> hyponyms for https://en-word.net/id/oewn-10251212-n ...
       done in 5.80s (4 rows)

Hypernyms:
  • adult male
  • crowned head
  • man
  • monarch
  • sovereign

Hyponyms:
  • King of England
  • King of France
  • King of Great Britain
  • King of the Germans

=== FEMALE MONARCH ===
Synset: https://en-word.net/id/oewn-10518940-n
    -> hypernyms for https://en-word.net/id/oewn-10518940-n ...
       done in 1.41s (1 rows)
    -> hyponyms for https://en-word.net/id/oewn-10518940-n ...
       done in 1.42s (1 rows)

Hypernyms:
  • female aristocrat

Hyponyms:
  • Queen of England

=== PRINCE ===
Synset: https://en-word.net/id/oewn-10492384-n
    -> hypernyms for https://en-word.net/id/oewn-10492384-n ...
       done in 6.74s (5 rows)
    -> hyponyms for https://en-word.net/id/oewn-10492384-n ...
       done in 15.49s 

This step converts the CSV-based results into a structured JSON format. The hypernym and hyponym label strings are split back into lists, preserving the full semantic structure for each term. In addition to the counts used for quantitative comparison, the JSON output retains all labels and relation metrics, making it suitable for qualitative analysis and inspection. This avoids recomputing the SPARQL queries while providing a more flexible and structured representation of the data

In [ ]:
import json
import pandas as pd

df = pd.read_csv("wordnet_gender_pairs_clean_sparql.csv")

structured_output = []

for _, row in df.iterrows():
    hypernyms = (
        row["hypernym_labels"].split(" | ")
        if pd.notna(row["hypernym_labels"]) and row["hypernym_labels"] != ""
        else []
    )

    hyponyms = (
        row["hyponym_labels"].split(" | ")
        if pd.notna(row["hyponym_labels"]) and row["hyponym_labels"] != ""
        else []
    )

    structured_output.append({
        "term": row["term"],
        "gender_side": row["gender_side"],
        "pair_male": row["pair_male"],
        "pair_female": row["pair_female"],
        "synset_uri": row["synset_uri"],

        "hypernym_count": int(row["hypernym_count"]),
        "hypernyms": hypernyms,

        "hyponym_count": int(row["hyponym_count"]),
        "hyponyms": hyponyms,

        "relation_predicate_count": int(row["wn_relation_predicate_count"]),
        "relation_edge_count": int(row["wn_relation_edge_count"]),
        "relation_both_directions": int(row["wn_relation_both_directions"]),
    })

with open("wordnet_term_details.json", "w", encoding="utf-8") as f:
    json.dump(structured_output, f, ensure_ascii=False, indent=2)

print("Saved to wordnet_term_details.json")


Processing male monarch...
    -> hypernyms for https://en-word.net/id/oewn-10251212-n ...
       done in 7.54s (5 rows)
    -> hyponyms for https://en-word.net/id/oewn-10251212-n ...
       done in 5.57s (4 rows)

Processing female monarch...
    -> hypernyms for https://en-word.net/id/oewn-10518940-n ...
       done in 3.13s (1 rows)
    -> hyponyms for https://en-word.net/id/oewn-10518940-n ...
       done in 1.81s (1 rows)

Processing prince...
    -> hypernyms for https://en-word.net/id/oewn-10492384-n ...
       done in 7.11s (5 rows)
    -> hyponyms for https://en-word.net/id/oewn-10492384-n ...
       done in 15.10s (11 rows)

Processing princess...
    -> hypernyms for https://en-word.net/id/oewn-10493649-n ...
       done in 6.99s (5 rows)
    -> hyponyms for https://en-word.net/id/oewn-10493649-n ...
       done in 6.68s (5 rows)

Processing emperor...
    -> hypernyms for https://en-word.net/id/oewn-10072812-n ...
       done in 4.13s (3 rows)
    -> hyponyms for https://e